In [7]:
import pandas as pd
from dotenv import load_dotenv
import os
import time
from openai import OpenAI


In [8]:
data_high = pd.read_json("../data/llm_formatted/high_llm.json")
data_low = pd.read_json("../data/llm_formatted/low_llm.json")
TARGET = "HubSpot"


In [9]:
data = pd.concat([data_high, data_low], axis=0)

In [10]:
data.head()

,id,subreddit_name,clean_title,clean_selftext,comments,num_of_comments,score,upvote_ratio,created_utc,thread_for_llm
0,1sgkv1i,hubspot,looking for feedback: an app for importing mes...,looking for feedback for an app that solves a ...,"[{'id': 'of5r0tx', 'text': 'The import tool in...",2,1,1.0,1775727274,SUBREDDIT: hubspot\n\nPOST [post_id_1sgkv1i]\n...
1,1sggfhk,hubspot,Spent two years optimizing our HubSpot setup. ...,Look. The workflows were fine. The sequences w...,"[{'id': 'of56vbz', 'text': 'The issue wasn’t y...",4,0,0.5,1775711551,SUBREDDIT: hubspot\n\nPOST [post_id_1sggfhk]\n...
2,1sg7tab,hubspot,Deal vs Projects? Where should I store contrac...,"We close Deals per service , and each engageme...","[{'id': 'of33v3v', 'text': 'Keep an eye on spr...",9,4,1.0,1775687736,SUBREDDIT: hubspot\n\nPOST [post_id_1sg7tab]\n...
3,1sg6pct,hubspot,Inventory Tracking,"Hey folks, I’m working on building an inventor...",[],0,1,1.0,1775685036,SUBREDDIT: hubspot\n\nPOST [post_id_1sg6pct]\n...
4,1sfwbbl,hubspot,Looking for feedback on my Hubspot Workflow app.,I need some honest feedback on a tool that I'v...,"[{'id': 'of0tk1k', 'text': 'Ok I’m in. If I li...",8,1,1.0,1775662489,SUBREDDIT: hubspot\n\nPOST [post_id_1sfwbbl]\n...


In [11]:
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [12]:
def llm_sentiment(thread, target, subreddit):

    prompt = f'''
        You are analyzing Reddit threads about companies.

        Target company: {target}
        Subreddit: {subreddit}
        Thread: {thread}

        Your task is to classify the overall sentiment of the thread toward {target} as: "positive", "negative", or "neutral".
        If the thread is from a subreddit dedicated to the company, assume it is about {target}, even if the name is not directly mentioned.
        Focus on what people in the thread actually think about {target}, not just the general tone.
        Use the full thread (post + comments). If opinions differ, base your answer on the overall dominant sentiment.

        Use these guidelines:
        - "positive" - users are clearly satisfied, happy, or say something good about {target}
        - "negative" - users complain, express frustration, or describe problems related to {target}
        - "neutral" - the thread mainly asks questions, shares information, or has no clear opinion

        Questions are usually neutral, unless they clearly describe a problem or frustration.  
        A positive or enthusiastic tone alone does not necessarily mean positive sentiment.

        If there is no clear opinion about {target}, choose "neutral".

        Also identify all relevant labels from the list below. You may return multiple labels or an empty list [].
        Only assign a label if there is clear evidence in the thread.

        Labels:
        - "bug_report": something is broken or not working
        - "limitation": works but lacks features or does not meet needs
        - "comparison": mentions competitors or alternative tools
        - "churn_risk": user is considering leaving or has already switched away
        - "purchase_intent": user is considering starting to use the product

        Write a short 1-2 sentence summary of the thread in Estonian.

        Return only this JSON:
        {{
            "sentiment": "positive" | "negative" | "neutral",
            "confidence": 0.0,
            "labels": [],
            "summary_et": "lühike kokkuvõte eesti keeles"
        }}
        '''
    
    try: 
        response = client.responses.create(
            model="gpt-4.1-mini",
            input=prompt,
        )

        raw = response.output_text.strip()
        raw = raw.replace("```json", "").replace("```", "").strip()

        analysis = json.loads(raw)
        return analysis
    
    except json.JSONDecodeError as e:
        print("JSON error on:" + response.output_text)
        return{
            "sentiment": "neutral",
            "sentiment_confidence": 0.0,
            "labels": [],
            "summary_et": "ERROR: JSON error"
        }


In [13]:
rows = []

start_time = time.time()

for i, row in data.iterrows():
    thread = row["thread_for_llm"]
    subreddit = row["subreddit_name"]

    #['id', 'subreddit_name', 'clean_title', 'clean_selftext', 'comments', 'num_of_comments', 'score', 'upvote_ratio', 'created_utc','thread_for_llm']

    out = {
        "id": row.get("id"),
        "subreddit_name": row.get("subreddit_name"),
        "num_of_comments" : row.get("num_of_comments"),
        "score": row.get("score"),
        "upvote_ratio" : row.get("upvote_ratio"),
        "created_utc": row.get("created_utc"),
        "thread_for_llm": thread,
    }

    try:
        llm = llm_sentiment(thread, TARGET, subreddit)

        out.update({
            "llm_sentiment": llm.get("sentiment", "neutral"),
            "llm_sentiment_confidence": llm.get("sentiment_confidence", 0.0),
            "llm_labels" : llm.get("labels", []),
            "llm_summary_et": llm.get("summary_et", "")
        })

    except Exception as e:
        out.update({
            "llm_sentiment": "neutral",
            "llm_sentiment_confidence": 0.0,
            "llm_labels": [],
            "llm_summary_et": f"ERROR: {e}"
        })

    rows.append(out)

    if (i + 1) % 50 == 0 or (i + 1) == len(data):
        print(i+1)



results = pd.DataFrame(rows)

seconds = time.time() - start_time
print(f"Aega läks: {seconds:.2f} s")

50
100
150
200
250
300
350
400
450
500
550
600
650
700
750
800
850
900
950
50
100
150
200
250
Aega läks: 4150.28 s


In [14]:
import os
results = results.reset_index(drop=True)

os.makedirs("../data/thread_level_results", exist_ok=True)
results.to_json("../data/thread_level_results/thread_results.json", orient="records", indent=2, force_ascii=False)
results.to_csv("../data/thread_level_results/thread_results.csv",index=False,encoding="utf-8")